In [2]:
import torch
print(f"CUDE available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDE available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
VRAM: 17.1 GB


In [49]:
import json

with open("/workspace/data/rlm_training_v2.json") as f:
    dataset = json.load(f)

print(f"Total examples: {len(dataset)}")
print(f"First example has {len(dataset[0]['messages'])} messages")

for msg in dataset[0]["messages"]:
    role = msg["role"]
    content = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"] 
    print(f"\n[{role}]: {content}")

Total examples: 155
First example has 4 messages

[user]: You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must S...

[assistant]: I'll search for the keyword 'patrol' to find relevant passages.

```python
idx = context.find('patrol')
print(f'Found at...

[user]: Output:
Found at position: 7111
patrols to; whether he thinks that the

 Picquet has been well posted, at night as well ...

[assistant]: FINAL(The manual instructs that patrols should be sent out at irregular intervals along routes that cover the front and ...


In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct") #loads the tokenizer

In [16]:
one_example = dataset[0]["messages"]
print(one_example)
print(len(tokenizer.apply_chat_template(one_example, tokenize = True)))
tokenizer.apply_chat_template(one_example, tokenize = False)

[{'role': 'user', 'content': 'You are a SEARCH assistant with a Python REPL. You search documents - nothing else.\n\nOUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.\n\nCONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.\n- Answering without searching = WRONG\n- Explaining instead of searching = WRONG\n- Any text before your code block = WRONG\n\nTOOLS:\n- `context` - the document (already loaded, DO NOT redefine)\n- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk\n\nWORKFLOW:\n1. Write ```python with print() to search\n2. STOP immediately after code block\n3. Wait for output (appears in next message)\n4. Search more OR give FINAL(answer)\n\nWhen done searching, end with: FINAL(your evidence-based answer)\n\n---\n\nWhat are the instructions for establishing a patrol route?'}, {'role': 'assistant', 'content': "I'll search for the keyword 'patrol' to find relevant passages.\n\n```python\

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Feb 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nYou are a SEARCH assistant with a Python REPL. You search documents - nothing else.\n\nOUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.\n\nCONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.\n- Answering without searching = WRONG\n- Explaining instead of searching = WRONG\n- Any text before your code block = WRONG\n\nTOOLS:\n- `context` - the document (already loaded, DO NOT redefine)\n- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk\n\nWORKFLOW:\n1. Write ```python with print() to search\n2. STOP immediately after code block\n3. Wait for output (appears in next message)\n4. Search more OR give FINAL(answer)\n\nWhen done searching, end with: FINAL(your evidence-based answer)\n\n---\n\nWhat are the instru

In [33]:
# full version for all 155 examples
lengths = []
for example in dataset:
    lengths.append(len(tokenizer.apply_chat_template(example["messages"], tokenize = True)))
print(lengths)

[478, 349, 480, 462, 343, 327, 334, 496, 478, 414, 453, 418, 463, 421, 443, 382, 421, 500, 423, 467, 438, 454, 455, 443, 324, 539, 403, 374, 415, 546, 406, 419, 523, 530, 409, 505, 489, 520, 594, 430, 405, 485, 555, 544, 433, 478, 479, 478, 474, 484, 477, 473, 463, 470, 475, 423, 547, 444, 466, 586, 442, 521, 575, 597, 673, 551, 601, 667, 652, 692, 524, 653, 430, 439, 544, 628, 592, 442, 553, 446, 555, 596, 482, 492, 397, 554, 466, 554, 580, 394, 583, 406, 419, 394, 428, 613, 395, 529, 372, 536, 616, 601, 515, 502, 643, 617, 399, 487, 624, 522, 396, 560, 570, 406, 479, 503, 526, 477, 534, 486, 436, 572, 446, 527, 515, 422, 489, 444, 507, 656, 698, 674, 701, 636, 530, 651, 711, 613, 756, 326, 325, 609, 515, 342, 510, 499, 463, 377, 485, 405, 404, 405, 560, 747, 642]


In [40]:
import numpy as np
data = np.array(lengths)
print(np.min(data))
print(np.max(data)) # the result of 756 tells us that max_seq_length can easily be 1024.
print(np.median(data 

324
756
485.0


In [45]:
import unsloth
from unsloth import FastLanguageModel
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Llama-3.2-1B-Instruct", max_seq_length, load_in_4bit = True)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


What that code just did was first imports the base model, Llama 3.2 1B-instruct, then states how long of a conversation the base model can handle at once. In this case it's 1024 because we first calculated every single one of our examples to realize that the largest it got was 756 tokens, so we have plenty of space. Finally load_in_4bit is basically the start to QLoRA, which basically only gives a limited number of adjustable weights since our computing power isnt exactly good enough for Full Fine Tuning. \
\
For this, we will useu get_peft_model() 

In [46]:
model = FastLanguageModel.get_peft_model(model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], lora_alpha = 16, lora_dropout = 0) 

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 16 layers with 16 QKV layers, 16 O layers and 0 MLP layers.


In [47]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


#### The SFT trainer needs a HuggingFace Dataset.

In [51]:
from datasets import Dataset
training_dataset = Dataset.from_list(dataset)
print(training_dataset)

Dataset({
    features: ['messages'],
    num_rows: 155
})


#### It's now time for setting up the trainer from the trl (Transformer Reinforcement Learning) library

In [56]:
from trl import SFTTrainer, SFTConfig

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

training_dataset = training_dataset.map(format_example)
training_args = SFTConfig(
    per_device_train_batch_size = 4,
    num_train_epochs = 4,
    learning_rate = 2e-4,
    max_seq_length = 1024,
    output_dir = "/workspace/outputs",
    logging_steps = 5
)

trainer = SFTTrainer(
    model = model,
    train_dataset = training_dataset,
    tokenizer = tokenizer,
    args = training_args,
)

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/155 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [57]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 155 | Num Epochs = 4 | Total steps = 80
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 3,407,872 of 1,239,222,272 (0.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.537900
10,3.265200
15,2.746600
20,2.355000
25,2.005600
30,1.612400
35,1.376500
40,1.176500
45,1.180800
50,0.970200


TrainOutput(global_step=80, training_loss=1.6440347135066986, metrics={'train_runtime': 63.8716, 'train_samples_per_second': 9.707, 'train_steps_per_second': 1.253, 'total_flos': 1818499766845440.0, 'train_loss': 1.6440347135066986, 'epoch': 4.0})

#### Time to Test

In [59]:
FastLanguageModel.for_inference(model);

In [60]:
messages = [
    {"role": "user", "content": "You are a SEARCH assistant with a Python REPL. You search documents - nothing else.\n\nOUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.\n\nCONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.\n- Answering without searching = WRONG\n- Explaining instead of searching = WRONG\n- Any text before your code block = WRONG\n\nTOOLS:\n- `context` - the document (already loaded, DO NOT redefine)\n- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk\n\nWORKFLOW:\n1. Write ```python with print() to search\n2. STOP immediately after code block\n3. Wait for output (appears in next message)\n4. Search more OR give FINAL(answer)\n\nWhen done searching, end with: FINAL(your evidence-based answer)\n\n---\n\nWhat is the maximum occupancy for the building?"}
] # example it hasn't seen yet

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
outputs = model.generate(inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


```python
idx = context.find('building')
if idx == -1:
    idx = context.find('building')
print(f'idx: {idx}')
print(context[idx:idx+200])
```


looks good! the redundant find for building is weird but it knows to retry. This will be fixed with more training data/ another epoch.

In [61]:
model.save_pretrained("/workspace/outputs/1b/rlm_lora_v2")
tokenizer.save_pretrained("/workspace/outputs/1b/rlm_lora_v2")

('/workspace/outputs/1b/rlm_lora_v2/tokenizer_config.json',
 '/workspace/outputs/1b/rlm_lora_v2/special_tokens_map.json',
 '/workspace/outputs/1b/rlm_lora_v2/chat_template.jinja',
 '/workspace/outputs/1b/rlm_lora_v2/tokenizer.json')